# GameTheory-13e : Réviser un modèle d'adversaire -- changer d'avis en perdant le moins possible

**Navigation** : [<< 13d-Optimistic-CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) | [Index](README.md) | [13f-Ataraxos-Patron >>](GameTheory-13f-Ataraxos-Patron-Python.ipynb)

**Kernel** : .NET C# ; source primaire : [carnet Python versionné](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb). Z3 est invoqué par Microsoft.Z3 4.12.2 et le chargeur partagé ; les références bibliographiques sont héritées.

***

## Concept

Dans un jeu à information imparfaite, un agent borné porte des **croyances** sur son adversaire :
ce qu'il a observé, ce qu'il croit savoir de son comportement. Puis une évidence nouvelle arrive
qui **contredit** ces croyances -- l'adversaire vient de faire exactement ce que la base affirme
qu'il ne fait jamais. Trois attitudes sont possibles :

- **refuser** l'évidence et garder la base intacte : l'agent agit sur un modèle faux ;
- **tout réinitialiser** et repartir d'une croyance vide : l'information accumulée est perdue ;
- **réviser** : accepter l'évidence en renonçant au *minimum* de ce qu'il croyait.

La théorie AGM (Alchourron, Gardenfors, Makinson, 1985) formalise la troisième attitude --
la triade expansion / contraction / révision. La famille 13 l'a déjà croisée sans la nommer :
le safe subgame solving de 13b *répare* un blueprint sans détruire l'équilibre global ;
l'abstraction à dette bornée du 19 contrôle ce qu'une simplification peut coûter. Ici le geste
est épistémique : c'est la **croyance** que l'on contracte, pas la stratégie.

Ce carnet atteste l'opération *réviser une croyance* sur un substrat indépendant de l'attestation
logique classique ([Tweety-04-Belief-Revision](../SymbolicAI/Tweety/Tweety-04-Belief-Revision-Python.ipynb) :
opérateurs CrMas sur machine virtuelle Java). Ici, pas de JVM : la base de croyances décrit un
**adversaire dans un jeu répété**, la consistance et les noyaux sont décidés par le solveur SMT
**Z3**, l'ordre d'enracinement est **mesuré** par valeur prédictive sur une histoire tenue à part,
et les propriétés de la révision sont **testées sur une batterie** -- jamais simplement assertées.

Le témoin visé, en trois morceaux mesurés :

1. la **trace de révision** -- quels noyaux, quelle incision, quels atomes survivent ;
2. le **coût des attitudes rivales** -- refuser et réinitialiser, contre le type vrai, en gain cumulé ;
3. le **contrôle négatif** -- une évidence cohérente avec la base doit produire une expansion pure,
   c'est-à-dire zéro incision.


In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"
#load "../SymbolicAI/SMT/Z3-API/Z3NativeLoader.cs"
Z3NativeLoader.Register(typeof(Microsoft.Z3.Context).Assembly);
Console.WriteLine($"Z3 natif : {Microsoft.Z3.Version.FullVersion}");


Installed Packages Microsoft.Z3, 4.12.2

Z3 natif : 5.4.1.0


### Environnement opérationnel

Le solveur Z3 (`z3-solver`) est l'organe de consistance : chaque contrôle de cohérence et chaque
noyau passe par lui. Les intersections d'ensembles de types qui apparaissent ensuite servent
uniquement à *interpréter* ce que Z3 décide -- la décision reste au solveur.

## Section 1 -- Le banc : un jeu répété et une base de croyances sur l'adversaire

Un jeu répété à trois actions pour l'agent (*Sécuriser*, *Milieu*, *Attaquer*) et trois pour
l'adversaire, à somme nulle. L'adversaire est l'un de **huit types** comportementaux -- chaque
type est une politique qui réagit à l'historique des coups de l'agent. L'agent ne connaît pas
le type : il porte une **base de croyances** `B`, un ensemble fini d'atomes, chaque atome étant
une contrainte sur le type caché, héritée d'une observation passée ou d'un a priori théorique.
Les tours sont simultanés : la réponse adverse du tour `t` ne voit que les tours strictement
antérieurs.

In [2]:
using System;
using System.Linq;
using System.Collections.Generic;
using Microsoft.Z3;

public class BeliefGame
{
    public static readonly int[,] Gain = { { 1, 2, -1 }, { 3, 1, -4 }, { -2, 0, 2 } };
    public static readonly string[] Names = Enumerable.Range(1, 8).Select(i => "t" + i).ToArray();
    private Random rng = new Random(20261010);
    public readonly Context Ctx = new Context(new Dictionary<string, string> { ["unsat_core"] = "true" });
    public Dictionary<string, HashSet<string>> Sense = new();
    public Dictionary<string, double> Scores = new();
    public List<string> Order = new();
    public HashSet<string> Initial = new();
    public HashSet<string> Revised = new();
    public static readonly int[] H = Enumerable.Repeat(0, 7).Concat(new[] { 1 }).Concat(Enumerable.Repeat(0, 3)).Concat(new[] { 1 }).ToArray();
    public static readonly int[] H2 = { 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1 };
    public int BestAdversary(int action) => Enumerable.Range(0, 3).OrderBy(b => Gain[action, b]).First();
    public int DefaultAdversary() => Enumerable.Range(0, 3).OrderBy(b => Enumerable.Range(0, 3).Sum(a => Gain[a, b])).First();
    public int Policy(string name, IReadOnlyList<int> h)
    {
        switch (name)
        {
            case "t1":
                return h.Count == 0 ? DefaultAdversary() : BestAdversary(Enumerable.Range(0, 3).OrderByDescending(a => h.Count(x => x == a)).First());
            case "t2": return h.Contains(2) ? 2 : 0;
            case "t3": return 2;
            case "t4": return h.Count > 0 && h[h.Count - 1] == 1 ? 2 : 0;
            case "t5": return rng.Next(3);
            case "t6": return h.Count > 0 ? BestAdversary(h[h.Count - 1]) : DefaultAdversary();
            case "t7": return h.Count % 3;
            case "t8": return h.Count < 5 ? 0 : h[h.Count - 1] == 1 ? 2 : 0;
            default: throw new ArgumentException(name);
        }
    }
    public int[] Run(string name, int[] h) => Enumerable.Range(0, h.Length).Select(t => Policy(name, h.Take(t).ToArray())).ToArray();
    public void Seed(int seed) => rng = new Random(seed);
    public BoolExpr Types(IEnumerable<string> names) => Ctx.MkOr(names.OrderBy(x => x).Select(n => Ctx.MkBoolConst(n)).ToArray());
    public BoolExpr Unique()
    {
        var vars = Names.Select(n => Ctx.MkBoolConst(n)).ToArray();
        var clauses = new List<BoolExpr> { Ctx.MkOr(vars) };
        for (int i = 0; i < vars.Length; i++)
            for (int j = i + 1; j < vars.Length; j++)
                clauses.Add(Ctx.MkNot(Ctx.MkAnd(vars[i], vars[j])));
        return Ctx.MkAnd(clauses.ToArray());
    }
    public Status Check(Dictionary<string, HashSet<string>> sense, IEnumerable<string> atoms, HashSet<string> evidence)
    {
        using (var solver = Ctx.MkSolver())
        {
            solver.Assert(Unique(), Types(evidence));
            foreach (var a in atoms) solver.Assert(Types(sense[a]));
            return solver.Check();
        }
    }
    public HashSet<string> Admitted(Dictionary<string, HashSet<string>> sense, IEnumerable<string> atoms, HashSet<string> evidence)
    {
        var result = new HashSet<string>();
        foreach (var name in Names)
            if (Check(sense, atoms, new HashSet<string>(evidence.Where(n => n == name))) == Status.SATISFIABLE) result.Add(name);
        return result;
    }
    // Toute consistance est tranchée par Z3, l'énumération conserve les seuls minimaux.
    public List<HashSet<string>> Cores(Dictionary<string, HashSet<string>> sense, HashSet<string> evidence)
    {
        var atoms = sense.Keys.OrderBy(a => a).ToArray();
        var found = new List<HashSet<string>>();
        for (int size = 1; size <= atoms.Length; size++)
            for (int mask = 1; mask < (1 << atoms.Length); mask++)
            {
                var subset = atoms.Where((a, i) => (mask & (1 << i)) != 0).ToHashSet();
                if (subset.Count != size || found.Any(k => k.IsSubsetOf(subset))) continue;
                if (Check(sense, subset, evidence) == Status.UNSATISFIABLE) found.Add(subset);
            }
        return found;
    }
    public static HashSet<string> Incision(IEnumerable<HashSet<string>> cores, IReadOnlyList<string> order)
    {
        var cut = new HashSet<string>();
        foreach (var core in cores)
            if (!core.Overlaps(cut)) cut.Add(core.OrderBy(a => Array.IndexOf(order.ToArray(), a)).First());
        return cut;
    }
    public HashSet<string> Revision(Dictionary<string, HashSet<string>> sense, HashSet<string> evidence, List<string> order)
    {
        var cut = Incision(Cores(sense, evidence), order);
        return Admitted(sense, sense.Keys.Where(a => !cut.Contains(a)), evidence);
    }
    public int BestResponse(HashSet<string> believed, List<int> h)
    {
        if (believed.Count == 0) return 0;
        var values = new double[3];
        foreach (string name in believed.OrderBy(n => n))
        {
            Seed(20261010 + 97 * Array.IndexOf(Names, name));
            int b = Policy(name, h);
            for (int a = 0; a < 3; a++) values[a] += (double)Gain[a, b] / believed.Count;
        }
        return Enumerable.Range(0, 3).OrderByDescending(a => values[a]).First();
    }
    public double Play(string attitude, string truth = "t8", int turns = 200, int cutoff = 13)
    {
        var hist = new List<int>();
        double result = 0;
        for (int t = 0; t < turns; t++)
        {
            var believed = t < cutoff || attitude == "refus" ? Initial : attitude == "revise" ? Revised : Names.ToHashSet();
            int a = BestResponse(believed, hist);
            int b = Policy(truth, hist);
            result += Gain[a, b];
            hist.Add(a);
        }
        return result;
    }
}

Console.WriteLine("Moteur des huit politiques et des noyaux initialisé.");


Moteur des huit politiques et des noyaux initialisé.


In [3]:
// Les huit politiques sont interrogées ; t5 est réellement aléatoire.
var jeu = new BeliefGame();
var premiers = new Dictionary<string, HashSet<int>>();
var milieu = new Dictionary<string, bool>();
var reactif = new Dictionary<string, bool>();
var stochastique = new Dictionary<string, bool>();
Console.WriteLine("8 types adverses : BR-frequent, Punisseur, Toujours-attaque, Bluffeur, Aleatoire, Opportuniste, Cycle, Bluffeur-retarde");
foreach (var nom in BeliefGame.Names)
{
    premiers[nom] = Enumerable.Range(0, 8).Select(_ => jeu.Policy(nom, Array.Empty<int>())).ToHashSet();
    milieu[nom] = Enumerable.Range(0, 8).All(_ => jeu.Policy(nom, new[] { 1 }) == 2);
    reactif[nom] = Enumerable.Range(0, 3).Select(a => jeu.Policy(nom, new[] { a })).Distinct().Count() > 1;
    stochastique[nom] = Enumerable.Range(0, 16).Select(_ => jeu.Policy(nom, new[] { 0 })).Distinct().Count() > 1;
    Console.WriteLine($"{nom} : premiers={string.Join(",", premiers[nom])}, après Milieu={milieu[nom]}, réactif={reactif[nom]}, stochastique={stochastique[nom]}");
}


8 types adverses : BR-frequent, Punisseur, Toujours-attaque, Bluffeur, Aleatoire, Opportuniste, Cycle, Bluffeur-retarde


t1 : premiers=2, après Milieu=True, réactif=True, stochastique=False


t2 : premiers=0, après Milieu=False, réactif=True, stochastique=False


t3 : premiers=2, après Milieu=True, réactif=False, stochastique=False


t4 : premiers=0, après Milieu=True, réactif=True, stochastique=False


t5 : premiers=0,1,2, après Milieu=False, réactif=True, stochastique=True


t6 : premiers=2, après Milieu=True, réactif=True, stochastique=False


t7 : premiers=0, après Milieu=False, réactif=False, stochastique=False


t8 : premiers=0, après Milieu=False, réactif=False, stochastique=False


In [4]:
// L'histoire a douze tours ; la compatibilité du type aléatoire est sondée huit fois.
var compatible = new Dictionary<string, bool>();
foreach (var nom in BeliefGame.Names)
{
    int essais = stochastique[nom] ? 8 : 1;
    compatible[nom] = Enumerable.Range(0, essais).Any(_ => jeu.Run(nom, BeliefGame.H).All(a => a == 0));
}
jeu.Sense["b1"] = BeliefGame.Names.Where(n => reactif[n]).ToHashSet();
jeu.Sense["b2"] = BeliefGame.Names.Where(n => !premiers[n].SetEquals(new[] { 2 })).ToHashSet();
jeu.Sense["b3"] = BeliefGame.Names.Where(n => !milieu[n]).ToHashSet();
jeu.Sense["b4"] = BeliefGame.Names.Where(n => compatible[n]).ToHashSet();
jeu.Sense["b5"] = BeliefGame.Names.Where(n => !stochastique[n]).ToHashSet();
foreach (var pair in jeu.Sense) Console.WriteLine($"{pair.Key} : {string.Join(",", pair.Value.OrderBy(x => x))}");
jeu.Initial = jeu.Admitted(jeu.Sense, jeu.Sense.Keys, BeliefGame.Names.ToHashSet());
Console.WriteLine($"B cohérente, types admis : {string.Join(",", jeu.Initial)}");


b1 : t1,t2,t4,t5,t6


b2 : t2,t4,t5,t7,t8


b3 : t2,t5,t7,t8


b4 : t2


b5 : t1,t2,t3,t4,t6,t7,t8


B cohérente, types admis : t2


### Lecture du banc

Chaque atome pris seul est raisonnable ; leur conjonction est radicale : `B` n'admet plus que
**le punisseur `t2`**. L'atome décisif est `b4` -- seul `t2` reproduit exactement les douze tours
sans attaque de `h`. Et c'est précisément là que le banc est piégé : le vrai type, `t8`, aurait
*attaqué* au tour 9 de `h` (en réponse au Milieu du tour 8) -- l'histoire observée n'était donc pas compatible avec lui : la phase de prudence
s'achève avant le Milieu du tour 8. La base est cohérente, mesurée... et fausse. C'est la situation que
la Section 2 va faire éclater.

Deux remarques de méthode. Les sémantiques des atomes ne sont pas déclarées : elles sont
**mesurées** sur les politiques (le tableau des sondes de la cellule précédente). Et `b3` --
« il n'attaque jamais après un Milieu *isolé* » -- garde `t8` dans son champ : le bluffeur
retardé n'attaque pas sur un Milieu isolé (phase de prudence), seulement sur un Milieu qui
s'inscrit dans une histoire établie. Cette nuance de sonde fera toute la différence au moment
de l'incision.

## Exercice 1 : énumérer les noyaux d'une base de croyances réseau

Une admin porte six croyances sur un attaquant inconnu. Les types possibles sont `A1..A8`.
La base :

| Atome | Croyance | Types compatibles |
|---|---|---|
| `c1` | l'attaquant ne fait que de la reconnaissance | {A1, A2, A3, A5, A6} |
| `c2` | l'attaquant agit seulement la nuit | {A1, A3, A4, A7} |
| `c3` | l'attaquant évite les serveurs de production | {A2, A5, A6, A8} |
| `c4` | l'attaquant n'a pas de credential valide | {A1, A2, A5, A6, A7, A8} |
| `c5` | l'attaquant est interne au réseau | {A2, A4, A5, A6} |
| `c6` | l'attaquant ne persiste pas au redémarrage | {A1, A3, A5, A7, A8} |

L'évidence du jour : l'attaquant vient d'utiliser un **credential valide** -- les types
compatibles avec cette évidence sont {A3, A4}. Un **noyau** de la base (pour cette évidence)
est un sous-ensemble *minimal* d'atomes dont la conjonction avec l'évidence est insatisfaisante.

**Travail demandé** : énumérer tous les noyaux de cette base pour cette évidence, sous forme
de liste de listes d'atomes triées (`[['c1', 'c4'], ...]` par exemple). Vérifier la minimalité
de chacun.

In [5]:
// Exercice ouvert : énumérer les conflits minimaux, sans modifier les entrées.
Func<Dictionary<string, HashSet<string>>, HashSet<string>, List<HashSet<string>>> noyauxExercice = (sense, evidence) => null;
var cSense = new Dictionary<string, HashSet<string>>
{
    ["c1"] = new() { "A1", "A2", "A3", "A5", "A6" },
    ["c2"] = new() { "A1", "A3", "A4", "A7" },
    ["c3"] = new() { "A2", "A5", "A6", "A8" },
    ["c4"] = new() { "A1", "A2", "A5", "A6", "A7", "A8" },
    ["c5"] = new() { "A2", "A4", "A5", "A6" },
    ["c6"] = new() { "A1", "A3", "A5", "A7", "A8" }
};
var casNoyaux = new[]
{
    (cSense, new HashSet<string> { "A3", "A4" }),
    (new Dictionary<string, HashSet<string>> { ["a"] = new() { "x" }, ["b"] = new() { "y" }, ["c"] = new() { "x", "y" } }, new HashSet<string> { "x", "y" }),
    (new Dictionary<string, HashSet<string>> { ["a"] = new() { "x", "y" } }, new HashSet<string> { "x" })
};
List<HashSet<string>> OracleNoyaux(Dictionary<string, HashSet<string>> sense, HashSet<string> evidence)
{
    var atoms = sense.Keys.OrderBy(a => a).ToArray();
    var found = new List<HashSet<string>>();
    for (int size = 1; size <= atoms.Length; size++)
        for (int mask = 1; mask < 1 << atoms.Length; mask++)
        {
            var subset = atoms.Where((a, i) => (mask & (1 << i)) != 0).ToHashSet();
            if (subset.Count != size || found.Any(k => k.IsSubsetOf(subset))) continue;
            var possible = new HashSet<string>(evidence);
            foreach (var atom in subset) possible.IntersectWith(sense[atom]);
            if (possible.Count == 0) found.Add(subset);
        }
    return found;
}
var repNoyaux = casNoyaux.Select(c => noyauxExercice(c.Item1.ToDictionary(k => k.Key, k => new HashSet<string>(k.Value)), new HashSet<string>(c.Item2))).ToArray();
if (repNoyaux.All(r => r == null)) Console.WriteLine("Exercice 1 à compléter.");
else
{
    if (repNoyaux.Any(r => r == null)) throw new Exception("Réponse partielle");
    for (int i = 0; i < casNoyaux.Length; i++)
    {
        var expected = OracleNoyaux(casNoyaux[i].Item1, casNoyaux[i].Item2);
        if (repNoyaux[i].Count != expected.Count || expected.Any(k => !repNoyaux[i].Any(x => x.SetEquals(k)))) throw new Exception("Noyaux incorrects");
    }
    Console.WriteLine("Exercice 1 : sondes réussies.");
}


Exercice 1 à compléter.


## Section 2 -- L'évidence contredit : noyaux et fonction d'incision

Au tour 13, l'agent joue *Milieu* ; l'adversaire réplique **Attaquer**. C'est l'évidence
`phi` : *il attaque en réponse à mon Milieu* -- mesurée sur les huit politiques face à
l'historique réel `h` (une évidence est liée aux faits ; les atomes de `B`, eux, venaient de
sondes génériques). La base devient incohérente. La théorie AGM ne dit pas « tout jeter » :
elle **contracte**.

La contraction par noyaux (Alchourron & Makinson) procède en deux temps :

1. **énumérer les noyaux** de `B` pour `phi` : les sous-ensembles *minimaux* d'atomes dont
   la conjonction avec `phi` est insatisfaisable -- Z3 tranche de chaque candidat ;
2. **choisir une incision** : retirer au moins un atome de *chaque* noyau, en sacrifiant
   en priorité les atomes les *moins enracinés*.

L'enracinement (`epistemic entrenchment`) n'est pas déclaré ici : il est **mesuré**. Chaque
atome est évalué sur une seconde histoire `h2`, tenue à part, où le vrai type finit par se
révéler : un atome est falsifié sur un tour de `h2` s'il exclut *tous* les types capables
d'expliquer l'action observée. L'atome qui prédit le mieux ce qui s'est réellement passé est
le plus enraciné -- le dernier qu'on accepte de sacrifier.

In [6]:
// La nouvelle evidence est mesurée sur les politiques déterministes après h.
var phi = BeliefGame.Names.Where(n => !stochastique[n] && jeu.Policy(n, BeliefGame.H) == 2).ToHashSet();
var noyaux = jeu.Cores(jeu.Sense, phi);
Console.WriteLine($"phi : {string.Join(",", phi)}");
foreach (var noyau in noyaux) Console.WriteLine($"noyau : {string.Join(",", noyau.OrderBy(x => x))}");
using (var solver = jeu.Ctx.MkSolver())
{
    solver.Assert(jeu.Unique(), jeu.Types(phi));
    foreach (var a in jeu.Sense.Keys.OrderBy(a => a)) solver.AssertAndTrack(jeu.Types(jeu.Sense[a]), jeu.Ctx.MkBoolConst("hyp_" + a));
    var status = solver.Check();
    var core = solver.UnsatCore.Select(e => e.ToString().Replace("hyp_", "")).ToHashSet();
    Console.WriteLine($"Z3 : {status}, core={string.Join(",", core)}, noyau={noyaux.Any(k => k.SetEquals(core))}");
}


phi : t1,t3,t4,t6,t8


noyau : b4


noyau : b1,b3


Z3 : UNSATISFIABLE, core=b4, noyau=True


In [7]:
// Chaque atome est évalué sur les 30 tours de l'histoire tenue à part.
int[] ActionsH2(string name)
{
    jeu.Seed(20261010 + 97 * Array.IndexOf(BeliefGame.Names, name));
    return jeu.Run(name, BeliefGame.H2);
}
var observed = ActionsH2("t8");
foreach (var atom in jeu.Sense.Keys)
{
    int falsified = 0;
    for (int t = 0; t < observed.Length; t++)
    {
        var explains = BeliefGame.Names.Where(n => ActionsH2(n)[t] == observed[t]).ToHashSet();
        if (explains.Count > 0 && !explains.Overlaps(jeu.Sense[atom])) falsified++;
    }
    jeu.Scores[atom] = 1.0 - (double)falsified / observed.Length;
}
jeu.Order = jeu.Scores.OrderBy(kv => kv.Value).ThenBy(kv => -jeu.Sense[kv.Key].Count).Select(kv => kv.Key).ToList();
foreach (var a in jeu.Order) Console.WriteLine($"{a} = {jeu.Scores[a]:F3}");
Console.WriteLine("Ordre : " + string.Join(" < ", jeu.Order));


b4 = 0.500


b5 = 1.000


b1 = 1.000


b2 = 1.000


b3 = 1.000


Ordre : b4 < b5 < b1 < b2 < b3


In [8]:
var cut = BeliefGame.Incision(noyaux, jeu.Order);
jeu.Revised = jeu.Admitted(jeu.Sense, jeu.Sense.Keys.Where(a => !cut.Contains(a)), phi);
Console.WriteLine($"Incision : {string.Join(",", cut.OrderBy(x => x))} ; après : {string.Join(",", jeu.Revised)}");


Incision : b1,b4 ; après : t8


### Lecture des noyaux et de l'incision

Deux noyaux, de formes différentes -- c'est ce qui rend l'incision intéressante :

- `['b4']` est un noyau **singleton** : la compatibilité avec `h`, à elle seule, exclut tous
  les types de `phi` ;
- `['b1', 'b3']` est un noyau **par paire** : « il réagit à mes coups » et « il n'attaque jamais
  après un Milieu isolé » pris ensemble n'admettent plus aucun type qui attaque après Milieu --
  alors que chacun seul en admet encore (`t1`/`t4`/`t6` pour `b1`, `t8` pour `b3`).

Le contrôle croisé le montre proprement : le *core* de Z3 (`assert_and_track`) rend `['b4']` --
un noyau -- mais un seul ; l'énumération par taille croissante rend les deux. Le core répond
« voilà un conflit », l'énumération répond « voilà *tous* les conflits minimaux ».

L'enracinement mesuré ne départage que `b4` : falsifié sur les quinze tours d'attaque de `h2`
(score 0,500), quand les quatre autres atomes expliquent chaque tour (1,000). Les ex æquo se
départagent par l'informativité (un atome qui exclut plus de types est plus enraciné), d'où
l'ordre `b4 < b5 < b1 < b2 < b3`. L'incision tombe alors : `b4` (seul membre du premier noyau)
et, dans la paire, `b1` -- moins enraciné que `b3`. La révision `B * phi` admet exactement
**`t8`, le type vrai** : en sacrifiant la lecture de l'histoire et l'hypothèse de réactivité,
l'agent garde « n'attaque jamais après un Milieu isolé » et « déterministe » -- deux atomes
vrais de `t8` -- et l'évidence affine le reste. Le changement est chirurgical : rien de ce qui
survit n'était falsifié par `h2`.

In [9]:
var psi = jeu.Sense["b2"];
var negativeCores = jeu.Cores(jeu.Sense, psi);
Console.WriteLine($"Evidence cohérente : {negativeCores.Count} noyau(x), expansion {string.Join(",", jeu.Admitted(jeu.Sense, jeu.Sense.Keys, psi))}");
var theta = BeliefGame.Names.Where(n => premiers[n].SetEquals(new[] { 2 })).ToHashSet();
var forcedCores = jeu.Cores(jeu.Sense, theta);
Console.WriteLine($"Evidence opposée : incision {string.Join(",", BeliefGame.Incision(forcedCores, jeu.Order).OrderBy(x => x))}");


Evidence cohérente : 0 noyau(x), expansion t2


Evidence opposée : incision b2,b3,b4


### Lecture des contrôles négatifs

**(a) L'évidence cohérente ne coûte rien.** Aucun noyau pour `psi`, incision vide : la
« révision » dégénère en expansion, les types admis ne changent pas (`{t2}`). C'est la
propriété de parcimonie du changement de croyance : confirmer n'est pas réviser -- mesuré,
pas décrété.

**(b) L'ordre d'enracinement n'est pas un veto.** Face à l'évidence « il attaque d'emblée,
certainement », trois noyaux singletons surgissent (`b2`, `b3`, `b4`) : aucun choix n'existe,
et l'atome le *plus* enraciné (`b3`) tombe avec les autres. L'enracinement hiérarchise les
sacrifices quand des alternatives existent ; il ne protège jamais une croyance que chaque
noyau accuse séparément. La mesure le dit crûment : c'est un ordre de coût, pas un droit de
veto.

## Exercice 2 : implémenter la fonction d'incision sous ordre d'enracinement

La fonction d'incision reçoit la liste des noyaux et un ordre d'enracinement (du moins au plus
enraciné). Elle retourne un ensemble d'atomes qui **touche chaque noyau au moins une fois**,
en privilégiant les atomes les moins enracinés. La batterie de vérification contrôle les deux
propriétés attendues : couverture complète des noyaux, et choix glouton local : pour chaque noyau encore intact, couper son atome le moins
enraciné. Cet ordre de traitement ne garantit pas une incision globalement minimale.

In [10]:
// Exercice ouvert : incision gloutonne, avec une copie des données de chaque cas.
Func<List<HashSet<string>>, List<string>, HashSet<string>> incisionExercice = (cores, order) => null;
var casIncision = new[]
{
    (new[] { "ab" }, "ab", "a"),
    (new[] { "ab", "bc" }, "cba", "b"),
    (new[] { "ab", "ac" }, "cba", "bc"),
    (new[] { "a", "bc" }, "cba", "ac"),
    (new[] { "abc" }, "cba", "c")
};
var repIncision = casIncision.Select(c => incisionExercice(c.Item1.Select(s => s.Select(ch => ch.ToString()).ToHashSet()).ToList(), c.Item2.Select(ch => ch.ToString()).ToList())).ToArray();
if (repIncision.All(r => r == null)) Console.WriteLine("Exercice 2 à compléter.");
else
{
    if (repIncision.Any(r => r == null)) throw new Exception("Réponse partielle");
    for (int i = 0; i < casIncision.Length; i++)
    {
        var expected = casIncision[i].Item3.Select(ch => ch.ToString()).ToHashSet();
        if (!repIncision[i].SetEquals(expected) || casIncision[i].Item1.Any(k => !k.Any(ch => repIncision[i].Contains(ch.ToString())))) throw new Exception("Incision incorrecte ou non couvrante");
    }
    Console.WriteLine("Exercice 2 : sondes réussies.");
}


Exercice 2 à compléter.


## Section 3 -- Le témoin : réviser coûte moins que réinitialiser ou refuser

Le vrai type caché est `t8` (bluffeur retardé). À partir du tour 13 -- juste après l'évidence
qui a déclenché la révision -- l'agent joue 200 tours selon trois attitudes épistémiques :

- **révisé** (AGM) : meilleure réponse espérée contre les types admis par `B * phi` ;
- **reset naïf** : il ne croit plus rien, meilleure réponse contre les huit types à égalité ;
- **refus conservateur** : il garde `B` intact et fait comme si de rien n'était.

La meilleure réponse maximise le gain espéré contre la crédence uniforme sur les types encore
crus, l'action adverse du tour étant prédite par les politiques réelles (tours simultanés :
la prédiction ne dépend pas de l'action courante de l'agent, l'influence est décalée d'un tour).
Le tableau final est le témoin mesuré : ce que coûte chaque attitude, en unités du jeu.

In [11]:
// Trois attitudes, chacune jouée 200 tours avec une coupure au treizième.
foreach (var attitude in new[] { "revise", "reset", "refus" })
    Console.WriteLine($"{attitude} : {jeu.Play(attitude):F1}");

Console.WriteLine($"Révisé - reset : {jeu.Play("revise") - jeu.Play("reset"):F1}");
Console.WriteLine($"Révisé - refus : {jeu.Play("revise") - jeu.Play("refus"):F1}");


revise : 450.0


reset : 18.0


refus : -765.0


Révisé - reset : 432.0


Révisé - refus : 1215.0


### Lecture du témoin

Les trois attitudes, 200 tours contre le vrai bluffeur retardé :

| Attitude | Gain cumulé | Ce qui s'est passé |
|---|---|---|
| révisé (AGM) | **+450,0** | croit `t8` exactement : alterne Milieu (+3 sur la prudence) et Attaquer (+2 sur l'attaque) |
| reset naïf | **+18,0** | croit les 8 types à égalité : le générateur C# produit une autre réalisation de la politique aléatoire |
| refus | **-765,0** | croit toujours `t2` : prédit la prudence, joue Milieu -- et mange -4 par tour |

Le refus est le piège auto-confirmant : croire le punisseur fait jouer Milieu (la meilleure
réponse à sa prudence), et ce Milieu répété déclenche l'attaque du bluffeur réel, que la
croyance fausse continue d'ignorer. Le reset, lui, ne se fait pas piéger -- mais il jette
l'information : il ne sait pas non plus *exploiter*. La révision gardait deux atomes vrais de
`t8`, et c'est exactement ce qu'ils valent : **la valeur du changement minimal est de +432,0**
(révisé moins reset) et **le coût du refus d'évidence de +1215,0** (révisé moins refus) --
refuser de réviser coûte ici plus de deux fois la valeur de toute l'information accumulée.
La graine et le générateur C# diffèrent de NumPy : le témoin reset est une réalisation distincte, les deux autres politiques sont déterministes.


## Section 4 -- Les propriétés de la révision : la batterie de tests

La batterie ci-dessous **tire des bases aléatoires**, calcule la révision par noyaux et
incision, puis contrôle sept prédicats ensemblistes finis sur chaque cas. Ces mesures
ne démontrent pas les postulats AGM généraux. Un échec s'affiche sous son nom.

In [12]:
// Vingt bases finies : les sept prédicats reproduisent ceux de la source.
var random = new Random(20261010);
var predicates = new[] { "succes", "inclusion", "vacuite", "coherence", "extensionnalite", "sous-expansion", "super-expansion" };
var counts = predicates.ToDictionary(p => p, p => 0);
HashSet<string> Pick(int count) => BeliefGame.Names.OrderBy(_ => random.Next()).Take(count).ToHashSet();
for (int trial = 0; trial < 20; trial++)
{
    int count = random.Next(3, 7);
    var sense = Enumerable.Range(0, count).ToDictionary(i => "x" + i, _ => Pick(random.Next(3, 8)));
    var evidence = Pick(random.Next(1, 4));
    var order = sense.Keys.OrderBy(_ => random.Next()).ToList();
    var baseTypes = BeliefGame.Names.ToHashSet();
    foreach (var value in sense.Values) baseTypes.IntersectWith(value);
    var rev = jeu.Revision(sense, evidence, order);
    var union = baseTypes.Union(evidence).ToHashSet();
    var intersection = baseTypes.Intersect(evidence).ToHashSet();
    var other = Pick(random.Next(1, 4));
    var combined = evidence.Intersect(other).ToHashSet();
    var left = jeu.Revision(sense, combined, order);
    var right = rev.Intersect(other).ToHashSet();
    if (rev.IsSubsetOf(evidence)) counts["succes"]++;
    if (rev.IsSubsetOf(union)) counts["inclusion"]++;
    if (intersection.Count == 0 || rev.SetEquals(intersection)) counts["vacuite"]++;
    if (rev.Count > 0) counts["coherence"]++;
    if (rev.SetEquals(jeu.Revision(sense, new HashSet<string>(evidence), order))) counts["extensionnalite"]++;
    if (right.Count == 0 || left.SetEquals(right)) counts["sous-expansion"]++;
    if (right.Count == 0 || left.IsSubsetOf(right)) counts["super-expansion"]++;
}
foreach (var p in predicates) Console.WriteLine($"{p} : {counts[p]}/20");


succes : 20/20


inclusion : 20/20


vacuite : 20/20


coherence : 20/20


extensionnalite : 20/20


sous-expansion : 20/20


super-expansion : 20/20


### Lecture de la batterie

Sept propriétés, vingt bases aléatoires, **20/20 partout**. Deux lectures :

- la **cohérence** (le résultat n'est jamais vide) tient mécaniquement à la frappe complète
  des noyaux : si les atomes survivants excluaient encore tout `phi`, ils contiendraient un
  noyau intact -- contredisant l'incision ;
- la **vacuité** est testée lorsque l'évidence intersecte la base ; la **sous-expansion**
  est testée lorsque la révision intersecte la seconde évidence. Ce sont les conditions
  effectivement employées par ces deux prédicats finis.

Ces sept prédicats sont les contrôles finis effectivement implémentés, non une preuve générale des postulats AGM. La révision *itérée*
(Darwiche-Pearl : réviser deux fois n'est pas réviser la conjonction) n'est pas testée ici --
l'incision gloutonne n'engage rien sur elle, et c'est une limite assumée de ce carnet.

## Exercice 3 : mesurer le témoin contre un autre type vrai

Rejouer la partie de la Section 3 contre le type `t4` (bluffeur immédiat, sans phase de
prudence initiale) : le classement des trois attitudes change-t-il ? En particulier, le
**refus** coûte-t-il plus ou moins cher face à un bluffeur qui se révèle dès le premier
Milieu ? Retourner le dictionnaire `{'revise': ..., 'reset': ..., 'refus': ...}` des gains
cumulés.
En C#, compléter `partieContreT4(attitude, tours, coupure)` : le garde appelle les trois attitudes et plusieurs budgets.


In [13]:
// Exercice ouvert : rejouer contre t4, pour les trois attitudes et plusieurs budgets.
Func<string, int, int, double?> partieContreT4 = (attitude, tours, coupure) => null;
var casT4 = (from a in new[] { "revise", "reset", "refus" } from pair in new[] { (200, 13), (25, 4), (8, 8) } select (a, pair.Item1, pair.Item2)).ToArray();
var expectedT4 = casT4.Select(c => jeu.Play(c.a, "t4", c.Item2, c.Item3)).ToArray();
var repT4 = casT4.Select(c => partieContreT4(c.a, c.Item2, c.Item3)).ToArray();
if (repT4.All(r => r == null)) Console.WriteLine("Exercice 3 à compléter.");
else
{
    if (repT4.Any(r => r == null)) throw new Exception("Réponse partielle");
    for (int i = 0; i < casT4.Length; i++)
    {
        double value = repT4[i].Value;
        if (!double.IsFinite(value) || Math.Abs(value - expectedT4[i]) > 1e-9) throw new Exception("Gain incorrect");
    }
    Console.WriteLine("Exercice 3 : sondes réussies.");
}


Exercice 3 à compléter.


## À retenir

Ce carnet a attesté l'opération **réviser une croyance** sur un substrat indépendant de
l'attestation logique classique :

- la **première attestation** ([Tweety-04](../SymbolicAI/Tweety/Tweety-04-Belief-Revision-Python.ipynb))
  révisait des bases propositionnelles par les opérateurs CrMas de Tweety, sur JVM ;
- la **seconde** (celle-ci) révisait un modèle d'adversaire en jeu répété : consistance
  décidée par Z3, enracinement *mesuré* par valeur prédictive, postulats *testés* sur
  batterie aléatoire.

Les trois morceaux du témoin sont dans les sorties : la trace de révision (noyaux, incision,
survie des atomes), le coût mesuré des attitudes rivales (refus, reset), et le contrôle
négatif (l'évidence cohérente ne coûte rien). La limite assumée : l'espace de types est fini
et petit, et l'incision gloutonne n'engage rien sur la révision itérée (Darwiche-Pearl) --
l'étendre à des croyances structurées sur espaces continus reste hors de ce carnet.

Pour le tableau des opérations ([#12204](https://github.com/jsboige/CoursIA/issues/12204)),
la forme du témoin est maintenant établie deux fois, sur des substrats indépendants par
l'outil (JVM/Tweety contre Z3/jeux) et par le stimulus (bases propositionnelles contre
modèles d'adversaire) -- la statuation d'admission reste au processus de l'EPIC.

### Trois mesures à retenir

1. **Réviser est l'attitude la moins coûteuse.** Sur 200 tours contre le vrai type `t8`, le gain cumulé est de **+450,0** avec la révision, **+18,0** avec une réinitialisation et **−765,0** avec un refus d'évidence : le changement minimal vaut **+432,0**, et le refus coûte **1 215,0** de plus que la révision.
2. **Z3 tranche, l'enracinement mesure.** La base `B` est cohérente (`sat`) et n'admet que le type `t2` ; l'évidence `phi` fait apparaître deux noyaux, `['b4']` et `['b1','b3']`, et le core de Z3 (`assert_and_track`) rend `['b4']`. L'enracinement mesuré sur l'histoire tenue à part départage : `b4 = 0,500` contre **1,000** pour `b5`, `b1`, `b2` et `b3`.
3. **La révision change la conclusion, le refus la détruit.** L'incision sacrifie `b1` et `b4` et conserve `b2`, `b3`, `b5` ; les types admis après révision sont `['t8']` — le type vrai. Le contrôle négatif tient : une évidence cohérente avec la base produit une expansion pure (`['t2']`), et la batterie des sept propriétés passe **20/20** sur vingt bases aléatoires.


## Vérifiez votre compréhension

1. Sur le banc de 200 tours, quelles sont les trois attitudes possibles et quel gain cumulé chacune rapporte-t-elle face au vrai type ?
2. Pourquoi l'incision sacrifie-t-elle `b4` et `b1` plutôt que d'autres atomes ?
3. Que vérifie le contrôle négatif, qu'une simple révision ne vérifierait pas ?
4. Que prouve la batterie de vingt bases aléatoires ?

<details><summary>Réponses</summary>

1. Réviser : **+450,0** ; réinitialiser : **+18,0** ; refuser l'évidence : **−765,0**. La révision apporte le gain le plus élevé dans cette réalisation, et l'écart entre réviser et refuser vaut **1 215,0**.
2. Parce que l'incision doit toucher **chaque** noyau : `['b4']` est un noyau singleton, `['b1','b3']` un noyau par paire. `b4` est le moins enraciné (0,500 contre 1,000) — le contrôle croisé de Z3 rend d'ailleurs `['b4']` comme core — et `b1` est le seul atome qui casse le second noyau sans sacrifier `b3`, qui reste compatible avec `t8`.
3. Il prend une évidence **cohérente** avec la base, et exige qu'elle produise une expansion pure : l'ensemble des types doit rester `['t2']`, sans qu'aucun atome soit sacrifié. Une révision ordinaire n'a pas cette garantie. À l'inverse, l'évidence « il attaque d'emblée, certainement » impose, elle, l'incision de `['b2','b3','b4']`.
4. Que les sept propriétés de la révision — succès, inclusion, vacuité, cohérence, extensionnalité, sous-expansion, sur-expansion — tiennent sur vingt bases aléatoires de 3 à 6 atomes : **20/20** sur chacune des sept, soit cent quarante contrôles.

</details>


## Pour aller plus loin

- **Carnet suivant** : [GameTheory-13f — Ataraxos, le patron](GameTheory-13f-Ataraxos-Patron-Python.ipynb) — le même geste, en appris : un réseau de croyances au lieu d'une révision symbolique.
- **Dans la même série** : [GameTheory-13d — Optimistic CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) — la mise à jour de regret, et sa version optimiste ; [GameTheory-13b — Safe Subgame Solving](GameTheory-13b-Safe-Subgame-Solving-Python.ipynb) — la réparation d'un blueprint, autre forme de révision.
- **Lectures** : Alchourron, Gärdenfors & Makinson (1985), *On the Logic of Theory Change* — la triade expansion / contraction / révision ; Darwiche & Pearl (1994) — la révision itérée, dont la limite est assumée dans la conclusion de ce carnet.
- **Hors série** : [Tweety-04 — Belief Revision](../SymbolicAI/Tweety/Tweety-04-Belief-Revision-Python.ipynb) — la première attestation, sur bases propositionnelles et JVM.


***

**Navigation** : [<< 13d-Optimistic-CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) | [Index](README.md) | [13g-Ataraxos-Organe-Natif >>](GameTheory-13g-Ataraxos-Organe-Natif.ipynb)
